# Módulo 2 · Clase 4 — Ramas, merge y colaboración

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Bloque | Qué aprenderás |
|---|--------|----------------|
| 1 | Qué es una rama | Por qué existen y cuándo usarlas |
| 2 | `git branch` y `git switch` | Crear, listar y moverte entre ramas |
| 3 | `git merge` | Fusionar ramas: fast-forward y 3-way |
| 4 | Resolución de conflictos | Qué hacer cuando Git no puede fusionar solo |
| 5 | Git flow simplificado | La estrategia de ramas que usan los equipos |
| 6 | Pull Requests | El proceso de revisión de código |
| 7 | Práctica | Feature branches para el sistema de notas |

> 💡 **Idea central:** Las ramas permiten que varias líneas de trabajo coexistan sin interferirse. Cada funcionalidad, corrección o experimento vive en su propia rama hasta que está lista para integrarse.

---
## Configuración del entorno de demo

In [ ]:
import os, shutil, subprocess
from pathlib import Path

DEMO = Path.home() / "ramas_demo"

if DEMO.exists():
    shutil.rmtree(DEMO)

DEMO.mkdir()
os.chdir(DEMO)

# Configurar git
!git init -b main
!git config user.name "Alumno Curso"
!git config user.email "alumno@curso.io"

# Crear la base del proyecto
Path("app.py").write_text("""\
def saludar(nombre):
    return f"Hola, {nombre}!"

def sumar(a, b):
    return a + b
""", encoding="utf-8")

Path("config.py").write_text("""\
VERSION = "1.0.0"
DEBUG = False
""", encoding="utf-8")

!git add .
!git commit -m "feat: estructura inicial del proyecto"

# Segundo commit
Path("README.md").write_text("# Demo de Ramas\n", encoding="utf-8")
!git add README.md
!git commit -m "docs: añadir README"

print(f"\nRepo demo listo en: {Path.cwd()}")
!git log --oneline

---
## 1. ¿Qué es una rama?

Una **rama** es un puntero móvil a un commit. Cuando creas una rama, Git solo crea un puntero nuevo — no copia los archivos. Por eso crear ramas en Git es prácticamente instantáneo.

```
ANTES (solo main):
A ── B ── C
         ▲
       main (HEAD)

DESPUÉS de crear rama 'feature':
A ── B ── C
         ▲ ▲
       main  feature (HEAD)

DESPUÉS de un commit en 'feature':
A ── B ── C ── D
         ▲     ▲
       main  feature (HEAD)
```

**`HEAD`** es el puntero especial que indica «dónde estás ahora». Siempre apunta a la rama (o commit) activa.

**¿Para qué sirven las ramas?**

- Trabajar en una nueva funcionalidad sin tocar el código que funciona.
- Arreglar un bug urgente mientras desarrollas algo nuevo.
- Experimentar sin miedo — si no funciona, borras la rama.
- Colaborar: cada persona trabaja en su rama y luego se fusionan.

---
## 2. `git branch` y `git switch` — Gestionar ramas

In [ ]:
# Ver ramas existentes
print("=== Ramas locales ===")
!git branch                  # * marca la rama actual

print("\n=== Ramas con info de tracking ===")
!git branch -vv

print("\n=== Todas las ramas (locales + remotas) ===")
!git branch -a

In [ ]:
# Crear una rama
!git branch feature/login
print("Ramas tras crear feature/login:")
!git branch

# Movernos a la nueva rama
!git switch feature/login
print("\nAhora estamos en:")
!git branch

In [ ]:
# Crear Y moverse en un solo paso (el más habitual)
!git switch -c feature/busqueda
print("\nRama creada y activada:")
!git branch

# Volver a main
!git switch main

# Eliminar una rama (cuando ya no la necesitas)
!git branch -d feature/busqueda  # -d: solo si ya está mergeada
# !git branch -D feature/busqueda # -D: forzar aunque no esté mergeada

print("\nRamas finales:")
!git branch

In [ ]:
# Hacer commits en la rama feature
!git switch feature/login

Path("login.py").write_text("""\
def login(usuario, contrasena):
    usuarios = {"admin": "secreto"}
    return usuarios.get(usuario) == contrasena
""", encoding="utf-8")

!git add login.py
!git commit -m "feat: añadir función de login"

Path("login.py").write_text("""\
USUARIOS = {"admin": "secreto", "usuario1": "clave"}

def login(usuario, contrasena):
    return USUARIOS.get(usuario) == contrasena

def logout(usuario):
    print(f"Adiós, {usuario}")
""", encoding="utf-8")

!git add login.py
!git commit -m "feat: añadir logout y diccionario de usuarios"

print("\nHistorial de feature/login:")
!git log --oneline

# Volver a main: login.py NO existe aquí
!git switch main
print("\nArchivos en main:")
!ls

---
## 3. `git merge` — Fusionar ramas

Cuando una rama está lista, la fusionas con `git merge`. Git tiene dos estrategias:

### Fast-forward merge

Ocurre cuando `main` no ha avanzado desde que se creó la rama. Git simplemente mueve el puntero hacia adelante — no hace falta un commit de merge.

```
ANTES:                          DESPUÉS (fast-forward):

A ── B ── C ── D                A ── B ── C ── D
         ▲     ▲                             ▲
       main  feature                   main + feature
```

### 3-way merge

Cuando ambas ramas han avanzado, Git crea un **merge commit** que une los dos historiales.

```
ANTES:                          DESPUÉS (3-way):

A ── B ── C ── E  (main)        A ── B ── C ── E ── M  (M = merge commit)
         └── D    (feature)              └── D ────┘
```

In [ ]:
# Fast-forward merge: main no ha avanzado
!git switch main

print("=== ANTES del merge ===")
!git log --oneline --all --graph

!git merge feature/login

print("\n=== DESPUÉS del merge (fast-forward) ===")
!git log --oneline --all --graph

print("\nAhora login.py existe en main:")
!ls

In [ ]:
# Preparar un 3-way merge: main y feature avanzan en paralelo

# Crear rama feature/reportes
!git switch -c feature/reportes
Path("reportes.py").write_text("def generar_reporte(): pass\n", encoding="utf-8")
!git add reportes.py
!git commit -m "feat: módulo de reportes (WIP)"

# Mientras tanto, main avanza también
!git switch main
Path("config.py").write_text("""\
VERSION = "1.1.0"
DEBUG = False
LOG_LEVEL = "INFO"
""", encoding="utf-8")
!git add config.py
!git commit -m "feat: añadir LOG_LEVEL en config"

print("Historial paralelo:")
!git log --oneline --all --graph

In [ ]:
# Merge 3-way: crea un merge commit
!git merge feature/reportes -m "Merge: integrar módulo de reportes"

print("Historial después del 3-way merge:")
!git log --oneline --all --graph

---
## 4. Resolución de conflictos

Un **conflicto** ocurre cuando dos ramas modifican la misma línea del mismo archivo. Git no puede decidir qué versión conservar — te lo muestra para que decidas tú.

```python
# Así marca Git un conflicto en el archivo:

<<<<<<< HEAD          # lo que tienes en tu rama actual
VERSION = "1.1.0"
=======               # separador
VERSION = "2.0.0"    # lo que viene de la rama que estás mergeando
>>>>>>> feature/v2
```

**Pasos para resolver:**
1. Abrir el archivo conflictivo.
2. Elegir qué conservar (o escribir algo diferente).
3. Eliminar los marcadores `<<<<<<<`, `=======`, `>>>>>>>`.
4. `git add <archivo>`.
5. `git commit`.

In [ ]:
# Provocar un conflicto: ambas ramas modifican config.py

# Rama feature/v2 cambia VERSION
!git switch -c feature/v2
Path("config.py").write_text("""\
VERSION = "2.0.0"
DEBUG = False
LOG_LEVEL = "INFO"
MAX_CONEXIONES = 100
""", encoding="utf-8")
!git add config.py
!git commit -m "feat: actualizar a versión 2.0.0"

# main también cambia config.py en la misma línea
!git switch main
Path("config.py").write_text("""\
VERSION = "1.2.0"
DEBUG = False
LOG_LEVEL = "DEBUG"
""", encoding="utf-8")
!git add config.py
!git commit -m "fix: activar log DEBUG para diagnóstico"

print("Historial con dos ramas divergentes:")
!git log --oneline --all --graph

In [ ]:
import subprocess

# Intentar merge: provocará conflicto
r = subprocess.run(["git", "merge", "feature/v2"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)

print("\nArchivo con marcadores de conflicto:")
print(Path("config.py").read_text())

print("\nEstado del repo durante el conflicto:")
!git status

In [ ]:
# Resolver el conflicto: elegimos VERSION 2.0.0 y LOG_LEVEL DEBUG
# (combinamos lo mejor de ambas ramas)
Path("config.py").write_text("""\
VERSION = "2.0.0"
DEBUG = False
LOG_LEVEL = "DEBUG"
MAX_CONEXIONES = 100
""", encoding="utf-8")

print("Archivo resuelto:")
print(Path("config.py").read_text())

# Marcar como resuelto y completar el merge
!git add config.py
!git commit -m "Merge: integrar v2 con corrección de log debug"

print("\nHistorial tras resolver el conflicto:")
!git log --oneline --all --graph

### Si el conflicto es demasiado complejo — abortar el merge

```bash
git merge --abort   # vuelves al estado anterior al intento de merge
```

Úsalo cuando necesitas tiempo para entender los cambios antes de resolver.

---
## 5. Git flow simplificado — Estrategia de ramas

Los equipos profesionales no trabajan en `main` directamente. Usan un flujo de ramas que protege el código estable.

### El modelo más común (Git flow simplificado)

```
main (producción — siempre estable)
 │
 ├── develop (integración — código listo para probar)
 │    ├── feature/login
 │    ├── feature/reportes
 │    └── feature/busqueda
 │
 └── hotfix/bug-critico (arreglos urgentes en producción)
```

### Flujo de trabajo diario

```bash
# 1. Crear rama desde develop (o desde main en proyectos pequeños)
git switch develop
git pull                          # asegurarse de tener lo último
git switch -c feature/mi-funcionalidad

# 2. Trabajar en la rama
# ... editar archivos ...
git add -p                        # añadir cambios selectivamente
git commit -m "feat: descripción"

# 3. Mantener la rama actualizada
git switch develop && git pull
git switch feature/mi-funcionalidad
git merge develop                 # o git rebase develop

# 4. Cuando está lista: Pull Request (no merge directo)
git push origin feature/mi-funcionalidad
# → abrir PR en GitHub

# 5. Después del merge en GitHub: limpiar
git switch develop && git pull
git branch -d feature/mi-funcionalidad
```

### Convención de nombres de ramas

| Prefijo | Para | Ejemplo |
|---------|------|---------|
| `feature/` | Nueva funcionalidad | `feature/login-social` |
| `fix/` | Corrección de bug | `fix/error-404-perfil` |
| `hotfix/` | Bug urgente en producción | `hotfix/token-expirado` |
| `docs/` | Solo documentación | `docs/actualizar-readme` |
| `refactor/` | Reestructurar sin cambiar funcionalidad | `refactor/auth-module` |

---
## 6. Pull Requests — Revisión de código

Un **Pull Request (PR)** es una solicitud para integrar una rama en otra. Su verdadero valor no es el merge técnico (eso Git lo hace solo) — es la **revisión de código** que ocurre antes del merge.

```
CICLO DE VIDA DE UN PULL REQUEST:

1. Desarrollador hace push de su rama a GitHub
   git push origin feature/login

2. Abre el PR en GitHub:
   - Base: develop (o main)
   - Compare: feature/login
   - Título y descripción clara
   - Asigna revisores

3. Revisores leen el código:
   - Comentan líneas concretas
   - Piden cambios (Request changes)
   - Aprueban (Approve)

4. Desarrollador corrige si hay feedback:
   git commit -m "fix: corregir según revisión"
   git push   (el PR se actualiza automáticamente)

5. Merge del PR (lo hace quien tiene permisos)

6. Borrar la rama en GitHub (botón tras el merge)
```

### Cómo escribir un buen PR

```markdown
## ¿Qué hace este PR?
Añade autenticación de usuarios con usuario y contraseña.

## ¿Por qué?
Necesitamos proteger las rutas del panel de administración (issue #42).

## ¿Cómo lo probé?
- [ ] Login correcto devuelve True
- [ ] Contraseña incorrecta devuelve False
- [ ] Usuario inexistente devuelve False

## Cambios
- `login.py`: nueva función `login(usuario, contrasena)` y `logout()`
- `config.py`: añadido `USUARIOS` (mover a DB en sprint 3)
```

---
## 7. Práctica — Feature branches para el sistema de notas

Aplicamos el flujo de trabajo profesional al repo `sistema_notas`.

---

### Ejercicio 1 — Gestión básica de ramas

**Enunciado:** Trabaja con el repo `sistema_notas`:

1. Crea tres ramas: `feature/exportar-csv`, `feature/busqueda-avanzada`, `fix/validacion-fecha`.
2. Muéstralas todas con `git branch`.
3. Escribe una función Python `estado_ramas(ruta)` que use `subprocess` para mostrar el nombre de la rama activa y la lista de todas las ramas locales.
4. Elimina la rama `fix/validacion-fecha` (sin hacer merge).
5. Muestra el historial con `--graph` para ver la estructura.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import os, subprocess
from pathlib import Path

NOTAS = Path.home() / "sistema_notas"
if NOTAS.exists():
    os.chdir(NOTAS)
    print(f"Trabajando en: {Path.cwd()}")
else:
    print("⚠️  Crea primero el repo sistema_notas")


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import os, subprocess
from pathlib import Path

NOTAS = Path.home() / "sistema_notas"
os.chdir(NOTAS)

# 1. Crear ramas
!git branch feature/exportar-csv
!git branch feature/busqueda-avanzada
!git branch fix/validacion-fecha

# 2. Ver todas
print("=== Ramas ===")
!git branch

# 3. Función estado_ramas
def estado_ramas(ruta):
    def git(args):
        r = subprocess.run(["git"] + args, cwd=str(ruta),
                           capture_output=True, text=True)
        return r.stdout.strip()

    activa = git(["branch", "--show-current"])
    todas = git(["branch"]).splitlines()

    print(f"\nRama activa: {activa}")
    print(f"Total de ramas: {len(todas)}")
    for rama in todas:
        print(f"  {rama.strip()}")

estado_ramas(NOTAS)

# 4. Eliminar rama (sin merge)
!git branch -D fix/validacion-fecha   # -D fuerza sin merge

# 5. Historial con graph
print("\nHistorial:")
!git log --oneline --all --graph
```
</details>

---
### Ejercicio 2 — Desarrollar una feature

**Enunciado:** Desarrolla la funcionalidad de exportar notas a CSV en su propia rama:

1. Muévete a `feature/exportar-csv`.
2. Crea el archivo `exportador.py` con una función `exportar_csv(notas, ruta_archivo)` que guarde las notas en formato CSV con columnas: `id,titulo,contenido,fecha`.
3. Commitea con mensaje apropiado (Conventional Commits).
4. Añade una función `importar_csv(ruta_archivo)` en el mismo archivo.
5. Commitea.
6. Fusiona `feature/exportar-csv` en `main` con `--no-ff` para forzar un merge commit aunque sea fast-forward.
7. Comprueba el historial gráfico tras el merge.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import os
from pathlib import Path

os.chdir(Path.home() / "sistema_notas")

# 1. Moverse a la rama
!git switch feature/exportar-csv

# 2. Crear exportador.py
Path("exportador.py").write_text("""\
import csv
from pathlib import Path

def exportar_csv(notas, ruta_archivo):
    ruta = Path(ruta_archivo)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "titulo", "contenido", "fecha"])
        writer.writeheader()
        for i, nota in enumerate(notas, 1):
            writer.writerow({
                "id": i,
                "titulo": nota.titulo,
                "contenido": nota.contenido,
                "fecha": str(nota.fecha),
            })
    return ruta
""", encoding="utf-8")

# 3. Primer commit
!git add exportador.py
!git commit -m "feat: exportar notas a CSV"

# 4. Añadir importar_csv
contenido = Path("exportador.py").read_text(encoding="utf-8")
contenido += """
def importar_csv(ruta_archivo):
    from nota import Nota
    notas = []
    with open(ruta_archivo, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for fila in reader:
            n = Nota(fila["titulo"], fila["contenido"])
            notas.append(n)
    return notas
"""
Path("exportador.py").write_text(contenido, encoding="utf-8")

# 5. Segundo commit
!git add exportador.py
!git commit -m "feat: importar notas desde CSV"

# 6. Merge con --no-ff (siempre crea merge commit, útil para el historial)
!git switch main
!git merge feature/exportar-csv --no-ff -m "Merge: integrar exportación CSV"

# 7. Historial gráfico
print("\nHistorial tras el merge:")
!git log --oneline --all --graph
```

**`--no-ff`** (no fast-forward) siempre crea un merge commit aunque no sea necesario. Esto es útil para mantener en el historial un punto claro donde se integró cada feature — facilita revertir features enteras si es necesario.
</details>

---
### Ejercicio 3 — Provocar y resolver un conflicto

**Enunciado:** Crea un conflicto controlado y resuélvelo:

1. Crea la rama `feature/nueva-config` desde `main`.
2. En esa rama, edita `gestor.py` añadiendo la constante `MAX_NOTAS = 100` al principio del archivo.
3. Commitea.
4. Vuelve a `main` y en `gestor.py` añade la constante `MAX_NOTAS = 50` al principio (misma línea, valor diferente).
5. Commitea en `main`.
6. Intenta mergear `feature/nueva-config` en `main`.
7. Resuelve el conflicto eligiendo `MAX_NOTAS = 200` (un valor definitivo diferente a los dos).
8. Completa el merge.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import os, subprocess
from pathlib import Path

os.chdir(Path.home() / "sistema_notas")

gestor_original = Path("gestor.py").read_text(encoding="utf-8")

# 1-3. Rama feature/nueva-config
!git switch -c feature/nueva-config
Path("gestor.py").write_text("MAX_NOTAS = 100\n" + gestor_original, encoding="utf-8")
!git add gestor.py
!git commit -m "feat: añadir límite de notas (100)"

# 4-5. main también modifica gestor.py
!git switch main
Path("gestor.py").write_text("MAX_NOTAS = 50\n" + gestor_original, encoding="utf-8")
!git add gestor.py
!git commit -m "feat: añadir límite de notas (50)"

# 6. Intentar merge — producirá conflicto
r = subprocess.run(["git", "merge", "feature/nueva-config"],
                   capture_output=True, text=True)
print(r.stdout + r.stderr)

# Ver el conflicto
print("Conflicto en gestor.py:")
print(Path("gestor.py").read_text()[:300])

# 7. Resolver: MAX_NOTAS = 200
contenido_resuelto = Path("gestor.py").read_text(encoding="utf-8")
# Eliminar todo el bloque de conflicto y poner el valor decidido
import re
contenido_limpio = re.sub(
    r'<<<<<<< HEAD.*?>>>>>>> feature/nueva-config\n',
    'MAX_NOTAS = 200\n',
    contenido_resuelto,
    flags=re.DOTALL
)
Path("gestor.py").write_text(contenido_limpio, encoding="utf-8")

# 8. Completar el merge
!git add gestor.py
!git commit -m "Merge: resolver conflicto MAX_NOTAS → 200"

print("\nHistorial final:")
!git log --oneline --graph
```
</details>

---
### Ejercicio 4 — Publicar una rama en GitHub

**Enunciado:** Practica el flujo completo de una feature branch publicada en GitHub:

1. Crea la rama `feature/busqueda-avanzada` en `sistema_notas`.
2. Añade en `gestor.py` un método `buscar_avanzada(termino, campo="titulo")` que filtre notas por `titulo` o `contenido` según el parámetro `campo`.
3. Commitea con mensaje convencional.
4. Haz push de la rama al remoto: `git push -u origin feature/busqueda-avanzada`.
5. Comprueba en GitHub que la rama aparece y GitHub te ofrece crear un Pull Request.
6. Escribe aquí (en una celda Markdown) la descripción del PR que escribirías.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


**✏️ Escribe aquí la descripción del PR:**

```markdown
## ¿Qué hace este PR?
...

## ¿Por qué?
...

## ¿Cómo lo probé?
- [ ] ...
```

---
### Ejercicio 5 (reto) — Analizador de ramas

**Enunciado:** Escribe la función `analizar_ramas(ruta)` que use `subprocess` para generar un informe completo del estado de las ramas de un repositorio:

- Rama activa
- Número de ramas locales y remotas
- Para cada rama local: nombre, hash del último commit, fecha, mensaje
- Qué ramas ya están mergeadas en `main` (candidatas a borrar)
- Qué ramas tienen commits que `main` no tiene (trabajo pendiente de integrar)

Pruébala con el repo `sistema_notas`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import subprocess
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import subprocess
from pathlib import Path

def analizar_ramas(ruta):
    ruta = Path(ruta).expanduser().resolve()

    def git(args):
        r = subprocess.run(["git"] + args, cwd=str(ruta),
                           capture_output=True, text=True)
        return r.stdout.strip()

    print(f"\n{'═'*60}")
    print(f"  INFORME DE RAMAS — {ruta.name}")
    print(f"{'═'*60}")

    # Rama activa
    activa = git(["branch", "--show-current"])
    print(f"\n  Rama activa: {activa}")

    # Contar ramas
    locales = [l.strip().lstrip("* ") for l in git(["branch"]).splitlines() if l.strip()]
    remotas = [r.strip() for r in git(["branch", "-r"]).splitlines() if r.strip()]
    print(f"  Ramas locales:  {len(locales)}")
    print(f"  Ramas remotas:  {len(remotas)}")

    # Detalle de cada rama local
    print(f"\n  {'RAMA':<30} {'HASH':<8} {'FECHA':<12} ÚLTIMO COMMIT")
    print(f"  {'-'*30} {'-'*8} {'-'*12} {'-'*25}")
    for rama in locales:
        info = git(["log", "-1", f"--format=%h|%ad|%s", "--date=short", rama])
        if info:
            partes = info.split("|", 2)
            hash_, fecha, msg = partes[0], partes[1], partes[2][:25]
        else:
            hash_, fecha, msg = "—", "—", "sin commits"
        marcador = " ◄" if rama == activa else ""
        print(f"  {rama:<30} {hash_:<8} {fecha:<12} {msg}{marcador}")

    # Ramas ya mergeadas en main
    mergeadas_raw = git(["branch", "--merged", "main"]).splitlines()
    mergeadas = [m.strip().lstrip("* ") for m in mergeadas_raw if m.strip()]
    candidatas = [m for m in mergeadas if m not in ("main", activa)]

    print(f"\n  RAMAS MERGEADAS EN main (candidatas a borrar):")
    if candidatas:
        for c in candidatas:
            print(f"    git branch -d {c}")
    else:
        print("    Ninguna")

    # Ramas con trabajo pendiente de integrar
    print(f"\n  RAMAS CON COMMITS PENDIENTES DE INTEGRAR EN main:")
    pendientes = False
    for rama in locales:
        if rama == "main":
            continue
        adelantados = git(["rev-list", "main.."+rama, "--count"])
        if adelantados and adelantados != "0":
            print(f"    {rama}: {adelantados} commit(s) adelante de main")
            pendientes = True
    if not pendientes:
        print("    Todas las ramas están integradas en main")

    print(f"\n{'═'*60}")


analizar_ramas("~/sistema_notas")
```
</details>

---
## Resumen de la clase

```bash
# ─── Ramas ─────────────────────────────────────────────────
git branch                        # listar ramas
git branch nombre                 # crear rama
git switch nombre                 # cambiar a rama
git switch -c nombre              # crear + cambiar
git branch -d nombre              # borrar (si está mergeada)
git branch -D nombre              # borrar (forzar)
git branch -vv                    # ver con info de tracking
git branch --merged main          # ramas ya integradas

# ─── Merge ─────────────────────────────────────────────────
git merge feature/login           # merge fast-forward (si es posible)
git merge feature/login --no-ff   # siempre crear merge commit
git merge --abort                 # abortar merge con conflictos

# ─── Conflictos ────────────────────────────────────────────
# 1. git status              → ver qué archivos tienen conflicto
# 2. editar el archivo       → quitar <<<, ===, >>> y elegir
# 3. git add <archivo>       → marcar como resuelto
# 4. git commit              → completar el merge

# ─── Ramas remotas ─────────────────────────────────────────
git push -u origin feature/login  # publicar rama en GitHub
git branch -r                     # ver ramas remotas
git fetch --prune                 # sincronizar ramas remotas eliminadas

# ─── Inspección ────────────────────────────────────────────
git log --oneline --all --graph   # historial de todas las ramas
git rev-list main..feature --count # cuántos commits adelante
```

## ¿Qué viene en la próxima clase?

La **Clase 5** es el proyecto integrador: versionamos el sistema de biblioteca de la Clase 12 del Módulo 1, aplicando todo lo aprendido — ramas, commits convencionales, historial limpio, tags de versión y publicación en GitHub.